# Hard Target-Shift Training: три отдельных запуска
Один объект → общий uniform D4 → clean и обязательный hard profile.
Loss = 0.5 CE(clean) + 0.5 CE(hard) + 3 JS, temperature=1.
Никакого AugMix или подмешивания original. Profile=(index+epoch)%6 (эпохи с 1).
Параметры из (seed, epoch, index), shared epoch виден persistent workers.

**A** ResNet50 с ImageNet V2, partial3/4, 24 эпохи: контроль стратегии.
**B** corrected AGMFA с ImageNet V2, partial3/4, 24 эпохи: стратегия + spatial architecture.
**C** fine-tune corrected AGMFA best.pt, partial2/3/4, 12 эпох: адаптация уже обученных features.
В C optimizer/scheduler новые; старые states не восстанавливаются.
Все запуски используют frozen backbone BN, seed42, одинаковые profiles.
Batch_size16 означает 16 объектов / 32 forward images. Val исходная, без hard/D4/TTA.
Test не читается. Старые модели/configs/results не меняются. Обучение выключено.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT/'src').is_dir():
    raise RuntimeError('Укажите ROOT — корень проекта с src и configs')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.experiment import load_config
from src.hard_target_shift import HardTargetViews, PROFILES, preview_profiles
from src.hard_target_training import run_hard_target_shift, plot_shift_history
from src.hard_target_checks import check_hard_target_shift
from src.hard_target_analysis import compare_shift_runs, shift_error_report
from src.utils import get_device


In [ ]:
CONFIGS = {
    'A': 'resnet50_hard_target_shift.json',
    'B': 'resnet50_agmfa_hard_target_shift.json',
    'C': 'resnet50_agmfa_hard_target_shift_finetune.json',
}
CHOICE = 'A'
RUN_TRAINING = False
RUN_PREFLIGHT = False
DEVICE = 'auto'
BATCH_SIZE = 16  # ORIGINAL objects ->32 images; reduce to8 if GPU OOM
NUM_WORKERS = 2
RAW_DIR = ROOT/'data/raw'  # На Kaggle укажите путь к train/val/classes.csv
cfg = load_config(ROOT/'configs'/CONFIGS[CHOICE])
cfg['data'].update(raw_dir=str(RAW_DIR), batch_size=BATCH_SIZE,
                   num_workers=NUM_WORKERS if get_device(DEVICE).type=='cuda' else 0)
cfg['training']['device'] = DEVICE
if get_device(DEVICE).type == 'cpu':
    torch.set_num_threads(4)
EXP_ID = cfg['exp_id']
print(EXP_ID, '| device:', get_device(DEVICE), '| paired images:', 2*BATCH_SIZE)


## Preview: train only
Один clean+D4 снимок и все шесть profiles с той же геометрией. Затем случайные
объекты с их назначенным profile. Preview не изменяет данные или глобальный RNG.
Если C: заранее разместите corrected `best.pt` по пути training.initialize_from.


In [ ]:
train_rows = pd.read_csv(RAW_DIR/'train.csv', dtype={'id': str})
PREVIEW_INDEX = 0
with Image.open(RAW_DIR/'train'/train_rows.iloc[PREVIEW_INDEX].id) as source:
    preview_profiles(source.convert('RGB'), cfg['data'], seed=cfg['seed'], index=PREVIEW_INDEX, epoch=1)
plt.show()

views = HardTargetViews(cfg['data'], cfg['seed'])
indices = np.random.default_rng(cfg['seed']).choice(len(train_rows), size=min(6,len(train_rows)), replace=False)
fig, axes = plt.subplots(len(indices), 2, figsize=(6, 2.7*len(indices)))
for row, index in enumerate(indices):
    with Image.open(RAW_DIR/'train'/train_rows.iloc[index].id) as source:
        clean, hard, trace = views.images(source.convert('RGB'), int(index), epoch=1)
    for axis, image in zip(axes[row], [clean, hard]):
        axis.imshow(image)
        axis.axis('off')
    axes[row,0].set_title(train_rows.iloc[index].id)
    axes[row,1].set_title(trace['profile'])
fig.tight_layout()
plt.show()


In [ ]:
if RUN_PREFLIGHT or RUN_TRAINING:
    report = check_hard_target_shift(cfg, project_root=ROOT, device=DEVICE)
    print(json.dumps(report, indent=2))
    if CHOICE == 'C' and report['initialization']['status'] != 'verified':
        raise RuntimeError('Для C нужен corrected AGMFA checkpoint; смотрите initialization в отчёте')
else:
    print('Preflight выключен. Установите RUN_PREFLIGHT=True для synthetic проверки.')


In [ ]:
if RUN_TRAINING:
    result = run_hard_target_shift(cfg, project_root=ROOT, show_plots=True)
    print('best accuracy:', result.val_accuracy, '| epoch:', result.best_epoch)
else:
    print('Обучение выключено: RUN_TRAINING=False')


## Сохранённые результаты и сравнение
`best.pt` — alias **best_accuracy.pt**. Также сохраняются **best_loss.pt** и last.pt.
Основные val.csv/val_probabilities.npz соответствуют best accuracy. Отдельно
сохранены val_best_loss.csv/NPZ и val_best_accuracy.csv/NPZ.
Оба критерия смотрим открыто; автоматической подгонки/ансамбля нет.
History включает total loss, clean/hard CE, unweighted JS, clean/hard accuracy и LR.
Profile counts — фактические counts каждого режима по эпохам.
Разница 1–2 val объектов сама по себе не подтверждает преимущество: смотрите
fixes/breaks, классы и конкретные low-quality/small-object случаи.


In [ ]:
logs = ROOT/'artifacts/logs'/EXP_ID
if (logs/'summary.json').is_file():
    display(pd.DataFrame([json.loads((logs/'summary.json').read_text())]))
    history = pd.read_csv(logs/'history.csv')
    plot_shift_history(history, show=True)
    display(pd.read_csv(logs/'profile_counts.csv').pivot(index='epoch', columns='profile', values='count'))
else:
    print('Этот run ещё не завершён.')


In [ ]:
BASELINES = ['resnet50_partial_domain_shift_stronger_001',
             'resnet50_agmfa_robust_augmix_fixed_001']
# Допишите val IDs маленьких/локальных объектов из ручного анализа:
SMALL_OBJECT_IDS = []
classes = pd.read_csv(RAW_DIR/'classes.csv')
if (ROOT/'artifacts/predictions'/EXP_ID/'val_probabilities.npz').is_file():
    comparison, cases = compare_shift_runs(ROOT, EXP_ID, BASELINES, classes, RAW_DIR)
    display(comparison)
    print('Quality proxies — только диагностика; low-detail может означать и однородную сцену.')
    shift_error_report(cases, EXP_ID, classes, RAW_DIR, manual_ids=SMALL_OBJECT_IDS)
else:
    print('Сравнение доступно после завершения выбранного run.')
